## Setting up data

In [72]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # go up one level from matching_model to code
sys.path.append(os.path.abspath('../..'))  # go up to root of the repository to read the files from source


In [73]:
import pandas as pd
from data_cleaning.parse_visitors import parse_visitor_data

# load raw csvs
visitors = pd.read_csv("../../source/visitors.csv")
answers = pd.read_csv("../../source/visitors_answers.csv")
questions = pd.read_csv("../../source/visitors_questions.csv")

In [74]:
# parse
visitor_data = parse_visitor_data(visitors, answers, questions)
visitor_data.head()

,email,gender,visitorId,stepId,questionId,answerValue,answerId,answerTypeId,answer,questionTypeId,question
14,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f730fd,5c8a78336d41a10da4f730fe,,5c8a78336d41a10da4f73100,Answer,To obtain general information,5bf7c399b82beb7a182cc3de,Reason for Attending the Event
20,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f73225,5c8a78336d41a10da4f73227,,5c8a78336d41a10da4f73244,Answer,Media,5bf7c399b82beb7a182cc3de,Which of the following best describes your job...
23,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f73252,5c8a78336d41a10da4f73253,,5c8a78336d41a10da4f73291,Answer,Travel Agent,5bf7c399b82beb7a182cc3de,Please indicate your company's main area of bu...
31,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f7336c,5c8a78336d41a10da4f7336d,,5c8a78336d41a10da4f73371,Answer,No influence,5bf7c399b82beb7a182cc3de,What role do you play in the purchasing decisi...
47,aleksandar.dimkov+mitt1_n5eA@bss.com.mk,M,67ada1ee197e604dd2722d1b,5c8a78336d41a10da4f730fd,5c8a78336d41a10da4f730fe,,5c8a78336d41a10da4f730ff,Answer,To source products and services,5bf7c399b82beb7a182cc3de,Reason for Attending the Event


## Mapping answers from visitors to known category classes

### Preparing question bank

To understand the categories of interest for the visitor, we can specifically look at the following questions - 
1. Please indicate your company's main area of business
2. Which of the following best describes your job function?

I inspected the answers for the question "Reason for Attending the Event" - the answers were very generic. I tried to see for each answer, the cohort of visitors, and analysed their answers to other questions; there is a lot of variance in these answers, so difficult to uniquely identify their category of interest from this question

Let's now try to map the answers to the above questions to already identified (customized) category classes.

In [75]:
question_bank = ["Please indicate your company's main area of business", "Which of the following best describes your job function?"]

business_type = visitor_data[visitor_data['question'] == question_bank[0]]['answer'].unique().tolist()
job_type = visitor_data[visitor_data['question'] == question_bank[1]]['answer'].unique().tolist()

In [76]:
business_type

['Travel Agent',
 'Event management',
 'Tour Operator',
 'IT solutions for travel industry',
 'Accommodation Provider']

The above business types have direct mapping to category classes, so we can uniquely identify their category of interest

In [77]:
job_type

['Media',
 'Sales',
 'Formation of tourist products',
 nan,
 'Visa support',
 'Guided tour services',
 'Marketing']

In [78]:
# Use this set of category_class_names (as values to identify the category of interest)
category_class_names = {
    "1": "Hotels & Stays",
    "2": "Tour Operators",
    "3": "Travel Agencies",
    "4": "Online Travel",
    "5": "Transport",
    "6": "Cars & Motorhomes",
    "7": "Glamping",
    "8": "Healthcare",
    "9": "Hotel Supplies",
    "10": "Parks",
    "11": "MICE",
    "12": "Tourism Offices",
    "13": "Travel Tech",
    "14": "Special Interest Travel",
    "15": "Media",
    "16": "Real Estate",
    "17": "Finance"
}



### Populating the category_class mapping from answers (to find category of interest)

We notice that some job types might not have unique category class mappings, so we do a deep dive on the visitors other answers.

The below job types are exluded from the interest map for the following reasons - (analysis performed below for each job type)
1. Sales - Everyone who has answered this as their job_type, have already indicated their business type as either Event management or Travel Agent, so we have covered them already
2. Formation of tourist products - Everyone who has answered this as their job_type, have already indicated their business type as either Accommodation Provider or Travel Agent, so we have covered them already

For the rest of the job types, thorough analysis was conducted to uniquely determine the category of interest

In [79]:
educational_interest_visitor_ids = visitor_data[visitor_data['answer'] == 'Sales']['visitorId'].unique().tolist()
visitor_data[(visitor_data['visitorId'].isin(educational_interest_visitor_ids)) & (visitor_data['question'].isin(question_bank))][['question', 'answer']]

,question,answer
53,Which of the following best describes your job...,Sales
54,Please indicate your company's main area of bu...,Event management
440,Which of the following best describes your job...,Sales
443,Please indicate your company's main area of bu...,Event management
499,Which of the following best describes your job...,Sales
500,Please indicate your company's main area of bu...,Travel Agent
638,Which of the following best describes your job...,Sales
639,Please indicate your company's main area of bu...,Travel Agent
733,Which of the following best describes your job...,Sales
736,Please indicate your company's main area of bu...,Event management


In [80]:
educational_interest_visitor_ids = visitor_data[visitor_data['answer'] == 'Formation of tourist products']['visitorId'].unique().tolist()
visitor_data[(visitor_data['visitorId'].isin(educational_interest_visitor_ids)) & (visitor_data['question'].isin(question_bank))][['question', 'answer']]

,question,answer
78,Which of the following best describes your job...,Formation of tourist products
81,Please indicate your company's main area of bu...,Travel Agent
258,Which of the following best describes your job...,Formation of tourist products
259,Please indicate your company's main area of bu...,Accommodation Provider
1141,Which of the following best describes your job...,Formation of tourist products
1144,Please indicate your company's main area of bu...,Travel Agent
1661,Which of the following best describes your job...,Formation of tourist products
1664,Please indicate your company's main area of bu...,Travel Agent
2281,Which of the following best describes your job...,Formation of tourist products
2282,Please indicate your company's main area of bu...,Accommodation Provider


In [81]:
educational_interest_visitor_ids = visitor_data[visitor_data['answer'] == 'Visa support']['visitorId'].unique().tolist()
visitor_data[(visitor_data['visitorId'].isin(educational_interest_visitor_ids)) & (visitor_data['question'].isin(question_bank))][['question', 'answer']]

,question,answer
140,Which of the following best describes your job...,Visa support
141,Please indicate your company's main area of bu...,Tour Operator
228,Which of the following best describes your job...,Visa support
230,Please indicate your company's main area of bu...,Tour Operator
469,Which of the following best describes your job...,Visa support
471,Please indicate your company's main area of bu...,Tour Operator
824,Which of the following best describes your job...,Visa support
825,Please indicate your company's main area of bu...,Tour Operator
854,Which of the following best describes your job...,Visa support
855,Please indicate your company's main area of bu...,Tour Operator


In [82]:
educational_interest_visitor_ids = visitor_data[visitor_data['answer'] == 'Guided tour services']['visitorId'].unique().tolist()
visitor_data[(visitor_data['visitorId'].isin(educational_interest_visitor_ids)) & (visitor_data['question'].isin(question_bank))][['question', 'answer']]

,question,answer
381,Which of the following best describes your job...,Guided tour services
383,Please indicate your company's main area of bu...,Travel Agent
945,Which of the following best describes your job...,Guided tour services
947,Please indicate your company's main area of bu...,Travel Agent
974,Which of the following best describes your job...,Guided tour services
976,Please indicate your company's main area of bu...,Travel Agent
2640,Which of the following best describes your job...,Guided tour services
2642,Please indicate your company's main area of bu...,Travel Agent
2788,Which of the following best describes your job...,Guided tour services
2790,Please indicate your company's main area of bu...,Travel Agent


In [83]:
educational_interest_visitor_ids = visitor_data[visitor_data['answer'] == 'Marketing']['visitorId'].unique().tolist()
visitor_data[(visitor_data['visitorId'].isin(educational_interest_visitor_ids)) & (visitor_data['question'].isin(question_bank))][['question', 'answer']]

,question,answer
529,Which of the following best describes your job...,Marketing
531,Please indicate your company's main area of bu...,Travel Agent
792,Which of the following best describes your job...,Marketing
794,Please indicate your company's main area of bu...,Travel Agent
1020,Which of the following best describes your job...,Marketing
1022,Please indicate your company's main area of bu...,Travel Agent
1417,Which of the following best describes your job...,Marketing
1419,Please indicate your company's main area of bu...,Travel Agent
1603,Which of the following best describes your job...,Marketing
1605,Please indicate your company's main area of bu...,Travel Agent


Therefore the below mapping provides the complete interest mapping to category

In [84]:
visitor_interest_map = {
    "Travel Agent": "Travel Agencies",
    "Event management": "MICE",
    "Tour Operator": "Tour Operators",
    "IT solutions for travel industry": "Travel Tech",
    "Accommodation Provider": "Hotels & Stays",
    "Media": "Media",
    #"Sales": "" 
    #"Formation of tourist products": "" 
    "Visa support": "Tour Operators", # inferred from above analysis
    "Guided tour services": "Travel Agencies", # inferred from above analysis
    "Marketing": "Travel Agencies" # inferred from above analysis
}

### Category class mapping

Mapping the interests, and defaulting to blank string, when the answer is not in the above category mapping

In [85]:
visitor_data['interest_category'] = visitor_data['answer'].apply(lambda x: visitor_interest_map.get(x, ""))

In [86]:
def fetch_unique_categories(x):
    category_list = x['interest_category'].tolist()
    category_set = set(category_list)
    category_set.discard("") # Discarding empty strings, if we didn't find any match in the interest mapping
    return category_set

In [87]:
visitor_interested_categories = visitor_data.groupby(['visitorId', 'email']).apply(lambda x: fetch_unique_categories(x)).reset_index(name='interest_category_list')
visitor_interested_categories.head()

C:\Users\sriram.c\AppData\Local\Temp\ipykernel_28288\455173958.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  visitor_interested_categories = visitor_data.groupby(['visitorId', 'email']).apply(lambda x: fetch_unique_categories(x)).reset_index(name='interest_category_list')


,visitorId,email,interest_category_list
0,0wcaegyyobblhhvfzwyibn0a,daniela.p+203_NfBj_lrIk@bss.com.mk,{Tour Operators}
1,1o5g2tlsho4gxzk3usr06z1c,sergey.usenko_ucwx_DPxA@ite.group,{}
2,35d97gdotgwa5lwpd0vs7k0h,tanja+182_jiPa_5IoL@bss.com.mk,{Travel Agencies}
3,3a17inawhgac2o0xtix3fb3n,aleksandar.dimkov+mitt1_n5eA_oG2a@bss.com.mk,{MICE}
4,3p80z1iocd67z0qvg8ju1cc0,aleksandar.dimkov+mitt10_V0iB_2bw2@bss.com.mk,{Tour Operators}


Great, we have now established the interest categories of visitors, we will handle no interest identified later

### Using the map_visitor_interests function in code/data_cleaning/

In [88]:
from data_cleaning.visitor_interest_mapping import map_visitor_interests

visitor_interested_categories = map_visitor_interests(visitor_data)
visitor_interested_categories.head()

,visitorId,email,interest_category_list
0,0wcaegyyobblhhvfzwyibn0a,daniela.p+203_NfBj_lrIk@bss.com.mk,{Tour Operators}
1,1o5g2tlsho4gxzk3usr06z1c,sergey.usenko_ucwx_DPxA@ite.group,{}
2,35d97gdotgwa5lwpd0vs7k0h,tanja+182_jiPa_5IoL@bss.com.mk,{Travel Agencies}
3,3a17inawhgac2o0xtix3fb3n,aleksandar.dimkov+mitt1_n5eA_oG2a@bss.com.mk,{MICE}
4,3p80z1iocd67z0qvg8ju1cc0,aleksandar.dimkov+mitt10_V0iB_2bw2@bss.com.mk,{Tour Operators}


## Note - recommending exhibitors by answers/visitor_email is the same. We will handle the recommendation with visitor email as input